# Hyperelasticity with Gaussian Processes

In [ ]:
# Import and initialize some stuff

import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from cycler import cycler
import seaborn as sns

# Set the color scheme
sns.set_theme()
colors = [
    "#0076C2",
    "#EC6842",
    "#A50034",
    "#009B77",
    "#FFB81C",
    "#E03C31",
    "#6CC24A",
    "#EF60A3",
    "#0C2340",
    "#00B8C8",
    "#6F1D77",
]
plt.rcParams["axes.prop_cycle"] = cycler(color=colors)

seed = 1
torch.manual_seed(seed)
np.random.seed(seed)

torch.set_default_dtype(torch.float64)

## A compact but complete GP code

You can find below a complete GP code with a the most popular kernel function (the squared exponential). Given a hyperparameter dictionary, this code performs the basic operations of a GP model:
- `se-kernel()`: Assembles covariance matrices based on the squared exponential kernel function. It has hyperparameters `sig_f` (kernel scaling) and `length` (length scale)
- `GP()`: Computes the posterior distribution of $y(\mathbf{x})$ at locations `X_hat` after observing targets `t` at locations `X`
- `GP_logmarglike()`: Computes the log marginal likelihood of a dataset `(X,t)`, can be used for hyperparameter tuning

In [ ]:
def se_kernel(X1, X2, **hyperparams):
    """
    :param X1: Some locations in input space [N1,D]
    :param X1: Other locations in input space [N2,D]
    :param hyperparams: The hyperparameters
    :return: a covariance matrix [N1,N2] based on the squared exponential kernel
    """
    sig_f = hyperparams["sig_f"]
    length = hyperparams["length"]

    out = sig_f**2 * torch.exp(-torch.cdist(X1, X2) ** 2 / (2 * length**2))

    return out

def GP(X, t, X_hat, kernel, hyperparams):
    """
    :param X: Observation locations [N,D]
    :param t: Observation values [N]
    :param X_hat: Prediction locations [Np,D]
    :param kernel: covariance function
    :param hyperparams: The hyperparameters
    :return: posterior mean [Np] and covariance matrix [Np,Np]
    """

    # Kernel of the observations
    k11 = kernel(X, X, **hyperparams)
    # Kernel of observations vs to-predict
    k12 = kernel(X, X_hat, **hyperparams)

    # Add jitter for numerical stability
    jitter = 1e-4 * torch.eye(k11.shape[0], dtype=k11.dtype, device=k11.device)

    # Algorithm 2.1 of https://gaussianprocess.org/gpml/chapters/RW.pdf
    C = k11 + hyperparams["noise"]**2 * torch.eye(k11.shape[0]) + jitter
    L = torch.linalg.cholesky(C)

    # Compute alpha = C_inv @ t using Cholesky decomposition
    alpha = torch.linalg.solve_triangular(L.T, torch.linalg.solve_triangular(L, t, upper=False), upper=True)

    # Compute posterior mean
    mu = torch.matmul(k12.T, alpha)

    # Compute v = L_inv @ k12 using Cholesky decomposition
    v = torch.linalg.solve_triangular(L, k12, upper=False)

    # Compute the posterior covariance
    cov = kernel(X_hat, X_hat, **hyperparams) - torch.matmul(v.T, v)

    return mu, cov

def GP_logmarglike(X, t, kernel, hyperparams):
    """
    Calculate the log marginal likelihood based on the observations (X, t) for a given kernel
    """
    # Kernel of the observations
    k11 = kernel(X, X, **hyperparams)

    # Add jitter for numerical stability
    jitter = 1e-4 * torch.eye(k11.shape[0], dtype=k11.dtype, device=k11.device)
    C = k11 + hyperparams["noise"] ** 2 * torch.eye(k11.shape[0]) + jitter
    L = torch.linalg.cholesky(C)
    logmarglike = (
        - 0.5 * torch.sum(torch.log(torch.diagonal(L)))
        - 0.5 * torch.linalg.norm(torch.linalg.solve_triangular(L, t, upper=False)) **2
        - 0.5 * len(t) * torch.log(torch.tensor(2 * torch.pi))
    )

    return logmarglike

## Functions for model selection (hyperparameter tuning)

You can use the code below to learn the hyperparameters of a GP model by maximizing the log marginal likelihood of a training dataset. Note that we do not need validation data to select a model! This is the core advantage of the Bayesian approach.

In [ ]:
def optimize(X, t, kernel, hyperparams_init):
    logsig = torch.tensor([hyperparams_init["sig_f"]]).requires_grad_(True)
    lognoise = torch.tensor([hyperparams_init["noise"]]).requires_grad_(True)
    loglen = torch.tensor([hyperparams_init["length"]]).requires_grad_(True)

    # We use the bfgs optimizer
    bfgs = torch.optim.LBFGS((logsig, lognoise, loglen), max_iter=50)
    def update():
        bfgs.zero_grad()

        hyperparams = {}
        hyperparams["sig_f"] = torch.exp(logsig)
        hyperparams["noise"] = torch.exp(lognoise)
        hyperparams["length"] = torch.exp(loglen)

        # Minime minus the likelihood, which means maximizing it
        loss = -GP_logmarglike(X, t, kernel, hyperparams) 
        loss.backward()
        return loss

    # We perform a single step. In this step, multiple iterations of the optimizer are performed
    bfgs.step(update)

    # Return the final log marginal likelihood and final hyperparameters
    hyperparams = {}
    hyperparams["sig_f"] = torch.exp(logsig)
    hyperparams["noise"] = torch.exp(lognoise)
    hyperparams["length"] = torch.exp(loglen)

    final_loss = GP_logmarglike(X, t, kernel, hyperparams)
    final_hps  = {'sig_f': torch.exp(logsig).detach()[0].item(), 'noise': torch.exp(lognoise).detach()[0].item(), 'length': torch.exp(loglen).detach()[0].item()}

    return final_loss, final_hps

def train_hyperparams(X, t, kernel, n_restarts=10):
    best_log_likelihood = -torch.inf
    best_hps = {}

    # Start the optimization from multiple initial points, for robustness
    for i in range(n_restarts):
        init_hps = {}
        init_hps['sig_f'] = torch.empty(1).uniform_(-5, 5).item()
        init_hps['noise'] = torch.empty(1).uniform_(-5, 5).item()
        init_hps['length'] = torch.empty(1).uniform_(-5, 5).item()
        
        try:
            current_log_likelihood, current_hps = optimize(X, t, kernel, init_hps)
        except:
            print(f"Optimization failed for run {i+1}/{n_restarts}. Skipping this run.")
            continue

        print(f"BFGS optimization {i+1}/{n_restarts}: Final log marginal likelihood for this run: {current_log_likelihood:.4f}, with hyperparameters {current_hps}")

        if current_log_likelihood > best_log_likelihood:
            best_log_likelihood = current_log_likelihood
            best_hps = current_hps
            print("Updated best likelihood and hyperparameters.")

    print("\n--- Best Optimization Result After Random Restarts ---")
    print(f"Best Log Marginal Likelihood: {best_log_likelihood:.4f}")
    print(f"Optimal sig_f: {best_hps['sig_f']:.4f}")
    print(f"Optimal noise: {best_hps['noise']:.4f}")
    print(f"Optimal length: {best_hps['length']:.4f}")

    return best_hps

## Fitting Bertoldi-Boyce hyperelasticity

Finally we use our self-contained GP implementation to fit hyperelastic material behavior. To keep it simple we will only fit a GP for stress component $P_{11}$ but using all four deformation gradient components as input features.

Because we are only predicting $P_{11}$, we can only compute four elements of the tangent stiffness, namely $\displaystyle\frac{\partial P_{11}}{\partial F_{kl}}$. But since we are using PyTorch we can enjoy its automatic differentiation functionalities and compute these tangent values without having to code anything extra.

### Getting a dataset

We load a big dataset but select only a handful of samples from it. One of your tasks will be to adjust this and see what happens. Note that we also normalize both inputs and outputs such that they are centered around zero with a unit standard deviation. This brings a lot of robustness to the hyperparameter optimizer and is standard procedure in machine learning

In [ ]:
# Define number of training samples
n_train_samples = 2

# Load the dataset and get all N samples
data = torch.load('BertoldiBoyce_dataset.pt')

F_tensor = data['F'].to(torch.float64)
P_tensor = data['P'].to(torch.float64)
D_tensor = data['D'].to(torch.float64)

# Reshape F to (N, 4)
F_tensor_flat = F_tensor.reshape(F_tensor.shape[0], -1)

# Take only the first row of D
D_tensor_flat = torch.stack((D_tensor[:,0,0,0,0], D_tensor[:,0,0,0,1], D_tensor[:,0,0,1,0], D_tensor[:,0,0,1,1]), dim=1)

# Extract P[0,0] and reshape to (N, 1)
P_tensor = P_tensor[:, 0, 0].reshape(-1, 1)

# Generate random permutation of indices
N_total = F_tensor.shape[0]
shuffled_indices = torch.randperm(N_total)

# Select only a few data points for training
train_indices = shuffled_indices[:n_train_samples]

# Create training data
F_train = F_tensor_flat[train_indices]
P_train = P_tensor[train_indices]
D_train = D_tensor_flat[train_indices]

# Compute mean and standard deviation for F_train and P_train to normalize the dataset
# This will improve model selection robustness. We can de-normalize later when predicting

F_mean = F_train.mean(dim=0, keepdim=True)
F_std = F_train.std(dim=0, keepdim=True)
F_train_normalized = (F_train - F_mean) / (F_std + 1e-8)

P_mean = P_train.mean(dim=0, keepdim=True)
P_std = P_train.std(dim=0, keepdim=True)
P_train_normalized = (P_train - P_mean) / (P_std + 1e-8)

### Learning the GP hyperparameters

Here we maximize our log marginal likelihood using the BFGS optimizer and the autograd capabilities of PyTorch. Feel free to adjust the number of optimizer restarts to see what happens.

In [ ]:
optimized_hyperparams = train_hyperparams(F_train_normalized, P_train_normalized, se_kernel, n_restarts=10)

print(f"\nOptimized Hyperparameters (Normalized Data): {optimized_hyperparams}")

### Predicting an unseen scenario: biaxial compression

After tweaking our prior with observations of random strain states, let's predict the well-defined case of biaxial compression (without shear). First we load the ground truth response for comparison:

In [ ]:
# Load the dataset and get all N samples
data = torch.load('BertoldiBoyce_compression.pt')

F_tensor = data['F'].to(torch.float64)
P_tensor = data['P'].to(torch.float64)
D_tensor = data['D'].to(torch.float64)

# Reshape F to (N, 4)
F_test = F_tensor.reshape(F_tensor.shape[0], -1).requires_grad_(True)

# Take only the first row of D
D_test = torch.stack((D_tensor[:,0,0,0,0], D_tensor[:,0,0,0,1], D_tensor[:,0,0,1,0], D_tensor[:,0,0,1,1]), dim=1)

# Extract P[0,0] and reshape to (N, 1)
P_test = P_tensor[:, 0, 0].reshape(-1, 1)

# Normalize the inputs (our GP prior has been trained to see this kind of data)
F_test_normalized = (F_test - F_mean) / (F_std + 1e-8)

Now that we have test locations `F_test`, a training dataset (`F_train_normalized`,`P_train_normalized`) and selected the model with `optimized_hyperparameters`, we can finally make new posterior predictions:

In [ ]:
mu_pred_normalized, cov_pred_normalized = GP(F_train_normalized, P_train_normalized, F_test_normalized, se_kernel, optimized_hyperparams)

# De-normalize the predictions to stresses
mu_pred = (mu_pred_normalized * P_std + P_mean).reshape(-1)
sig_normalized = torch.sqrt(torch.diag(cov_pred_normalized))
sig = (sig_normalized * P_std).reshape(-1)

# Use Automatic Differentiation to get the tangent stiffness for free
D_pred = torch.autograd.grad(outputs=mu_pred, inputs=F_test,
                             grad_outputs=torch.ones_like(mu_pred),
                             retain_graph=True, create_graph=False)[0]

Finally, we can plot how the model performs:

In [ ]:
def plot_gp_predictions(mu_pred, sig, F_test, P_test, D_pred, D_test, P_00_mean, P_00_std):
    # Ensure P_test is a flat tensor for plotting
    P_test_flat = P_test.flatten()

    # Create a figure with three subplots in a single row
    fig, axes = plt.subplots(1, 5, figsize=(24, 6))

    # --- Subplot 1: P_11 vs F_11 (GP predictions and ground truth) ---
    axes[0].plot(F_test[:, 0].detach().numpy(), mu_pred.detach().numpy(), color='C0', lw=2, label='GP Mean')
    axes[0].fill_between(
        F_test[:, 0].detach().numpy(),
        (mu_pred - 1.96 * sig).detach().numpy(),
        (mu_pred + 1.96 * sig).detach().numpy(),
        color='C0',
        alpha=0.15,
        label='95% Confidence Interval',
    )
    axes[0].plot(F_test[:, 0].detach().numpy(), P_test_flat.detach().numpy(), 'k--', label='Ground Truth')
    axes[0].set_xlabel('$F_{11}$', fontsize=13)
    axes[0].set_ylabel('$P_{11}$', fontsize=13)
    axes[0].set_title('GP Predictions vs. Ground Truth for P_11', fontsize=15)
    axes[0].legend(loc='upper left', fontsize=10)
    axes[0].grid(True)

    axes[1].plot(F_test[:, 0].detach().numpy(), D_pred[:, 0].detach().numpy(), color='C0', lw=2, label='GP (autograd of mean)')
    axes[1].plot(F_test[:, 0].detach().numpy(), D_test[:, 0].detach().numpy(), 'k--', label='Ground Truth')
    axes[1].set_xlabel('$F_{11}$', fontsize=13)
    axes[1].set_ylabel('$D_{1111}$', fontsize=13)
    axes[1].set_title('GP Predicted $D_{1111}$ vs. Ground Truth', fontsize=15)
    axes[1].legend(loc='upper left', fontsize=10)
    axes[1].grid(True)

    axes[2].plot(F_test[:, 0].detach().numpy(), D_pred[:, 1].detach().numpy(), color='C0', lw=2, label='GP (autograd of mean)')
    axes[2].plot(F_test[:, 0].detach().numpy(), D_test[:, 1].detach().numpy(), 'k--', label='Ground Truth')
    axes[2].set_xlabel('$F_{11}$', fontsize=13)
    axes[2].set_ylabel('$D_{1112}$', fontsize=13)
    axes[2].set_title('GP Predicted $D_{1112}$ vs. Ground Truth', fontsize=15)
    axes[2].legend(loc='upper left', fontsize=10)
    axes[2].grid(True)

    axes[3].plot(F_test[:, 0].detach().numpy(), D_pred[:, 2].detach().numpy(), color='C0', lw=2, label='GP (autograd of mean)')
    axes[3].plot(F_test[:, 0].detach().numpy(), D_test[:, 2].detach().numpy(), 'k--', label='Ground Truth')
    axes[3].set_xlabel('$F_{11}$', fontsize=13)
    axes[3].set_ylabel('$D_{1121}$', fontsize=13)
    axes[3].set_title('GP Predicted $D_{1121}$ vs. Ground Truth', fontsize=15)
    axes[3].legend(loc='upper left', fontsize=10)
    axes[3].grid(True)

    axes[4].plot(F_test[:, 0].detach().numpy(), D_pred[:, 3].detach().numpy(), color='C0', lw=2, label='GP (autograd of mean)')
    axes[4].plot(F_test[:, 0].detach().numpy(), D_test[:, 3].detach().numpy(), 'k--', label='Ground Truth')
    axes[4].set_xlabel('$F_{11}$', fontsize=13)
    axes[4].set_ylabel('$D_{1122}$', fontsize=13)
    axes[4].set_title('GP Predicted $D_{1122}$ vs. Ground Truth', fontsize=15)
    axes[4].legend(loc='upper left', fontsize=10)
    axes[4].grid(True)

    # Adjust layout to prevent overlapping titles/labels
    plt.tight_layout()

    # Display the plot
    plt.show()

# Call the new plotting function
plot_gp_predictions(mu_pred, sig, F_test, P_test, D_pred, D_test, P_mean, P_std)